In [1]:
# Part Two

# Scrape the content of https://www.lemonde.fr/Links to an external site. and save it as a CSV.

# We want: titles, subhead, article URL, whether it's premium or not, byline, article type, image URL.

# Bonus, if you want to get fancy:

# Make the CSV file auto-updating. Use this tutorial (videoLinks to an external site., textLinks to an external site.) but just ignore the visualization/datawrapper aspect
# Tips:

# Use the 02 - BBC Scraping.ipynb notebook as a guide for this one. We used .get('href', None) in class to avoid "the link doesn't exist" problems, but it uses try/except. Both are reasonable, but try/except is more flexible.
# You're going to be missing some article URLs - you should use try/except on them to avoid them being a problem, buuuut after that I would recommend printing the element to see why your approach isn't working (it's 100% possible to get all of the URLs!)
# If you want to make sure the element you're getting has a specific attribute, you can do something like .select_one("a[href]"). That will only give you anchor tags (links) that have hrefs.
# Instead of yes/no for the premium question, you can think of it as "put text in this column if the article is premium, don't put text in it if it is not"


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

response = requests.get("https://www.lemonde.fr/")
doc = BeautifulSoup(response.text, 'html.parser')

In [3]:
#what class are we looking at ? inspect the website
#we have: h1, div class="ds-card ", div class="ds-card__media", picture class="ds-card__media__picture"
#div class="ds-card "

#section class="ds-area ds-area--top". THe section class wouldn't work because it's too big, 
#you want to grab the structure that's right above what you are looking for (one level above). parent versus child
#you observe that each block that starts with a. div class = ds card, has all the elemnts you are looking for
#here, it starts with "a" because each article is essentially a link to another webpage
#you try with different things, (div_article, div_class, div_clicktype) and each type you count the number of elements inside
#and you have a look at it using this formula, to see if it spits out the thing you are looking for (byline, title, etc)


In [10]:
# count stories. find everything with div, look at number of elements. #find div class="ds-card "
#each one of these is going to be a row

stories = doc.find_all(class_="ds-card")
len(stories)



94

In [11]:
#print stories.
stories

[<div class="ds-card" data-chapo="true" data-components="['chapo']" data-display-editorial-type="false" data-first="true" data-linked-posts="true" data-live="false" data-live-is-open="false" data-live-posts="false" data-media="true" data-media-type="image" data-opinion="false" data-testimony="false" data-variant="l" data-zone="titles title_1"> <div class="ds-card__clickarea"> <div class="ds-card__media"> <picture class="ds-card__media__picture"> <img alt="Le PDG d’OpenAI Sam Altman, le président américain Donald Trump et le PDG de Google DeepMind Demis Hassabis et le président sud-coréen Lee Jae Myung lors d’un déjeuner lors du sommet du G7 à Évian-les-Bains, le 17 juin 2026. - KAMIL ZIHNIOGLU POUR « LE MONDE »" aria-hidden="true" decoding="async" fetchpriority="high" height="443" loading="eager" role="presentation" sizes="(min-width: 1024px) 556px, (min-width: 768px) 664px, 100vw" src="https://img.lemde.fr/2026/06/17/873/0/6517/4344/664/442/75/0/81b9ace_upload-1-kcyoxfu0rrqt-r5-4001.j

In [13]:
#This Python code is a web scraper (using the library Beautiful Soup). 
#Its job is to loop through a list of HTML news elements (stories), 
#extract specific pieces of information about each story, and save them into a structured format.

# Starting off without ANY rows
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        print("Headline not found!")
        continue
        
    try:
        # Find me a media__link OR a reel_link
        row['href'] = story.select_one('a')['href']
    except:
        print("Couldn't find a link")

    try:
        row['tag'] = story.select_one('.ewmvhB').text.strip()
    except:
        print("Couldn't find a tag!")

    try:
        row['tag'] = story.find(attrs={'data-testid': 'card-description'}).text.strip()
    except:
        print("Couldn't find a summary")

    print(row)
# When we're done adding info to our row, we're going to add it into our list
    # of rows

    rows.append(row)

----
Couldn't find a tag!
Couldn't find a summary
{'title': 'Au sommet du G7\xa0à Evian, les dirigeants appellent les entreprises de la tech à garantir la sécurité des mineurs en ligne', 'href': 'https://www.lemonde.fr/international/article/2026/06/17/au-g7-les-dirigeants-appellent-la-tech-a-garantir-la-securite-des-mineurs-en-ligne_6704245_3210.html'}
----
Couldn't find a tag!
Couldn't find a summary
{'title': 'Donald Trump menace de\xa0bombarder à\xa0nouveau l’Iran, si\xa0Téhéran «\xa0ne se comporte pas bien\xa0», à\xa0deux jours de\xa0la\xa0signature du\xa0protocole d’accord', 'href': 'https://www.lemonde.fr/international/live/2026/06/17/en-direct-guerre-au-moyen-orient-donald-trump-menace-de-bombarder-a-nouveau-l-iran-si-teheran-ne-se-comporte-pas-bien-a-deux-jours-de-la-signature-du-protocole-d-accord_6701050_3210.html'}
----
Couldn't find a tag!
Couldn't find a summary
{'title': '«\xa0Les incidents se multiplient\xa0» en mer Baltique, où la marine russe «\xa0opère maintenant de f

In [14]:
df = pd.DataFrame(rows)
df

,title,href
0,"Au sommet du G7 à Evian, les dirigeants appell...",https://www.lemonde.fr/international/article/2...
1,Donald Trump menace de bombarder à nouveau l’I...,https://www.lemonde.fr/international/live/2026...
2,« Les incidents se multiplient » en mer Baltiq...,https://www.lemonde.fr/international/article/2...
3,« Le Mondial vient de commencer : il oppose 48...,https://www.lemonde.fr/sport/article/2026/06/1...
4,La vague de chaleur va monter en intensité pen...,https://www.lemonde.fr/planete/article/2026/06...
...,...,...
89,"La vague de chaleur qui déferle sur la France,...",https://www.lemonde.fr/planete/article/2026/06...
90,"Eduardo Bolsonaro, fils de l’ex-président brés...",https://www.lemonde.fr/international/article/2...
91,Coupe du monde 2026 : une équipe de France à d...,https://www.lemonde.fr/sport/article/2026/06/1...
92,Argentine-Algérie : un Lionel Messi record lan...,https://www.lemonde.fr/sport/live/2026/06/17/e...


In [15]:
#saving as csv
df.to_csv("scrapping_le_Monde.csv")